# Maison LLF EDA
This notebook performs exploratory data analysis on `maison-llf-demographics.csv` and `maison-llf-features.csv`.

In [ ]:
# Import Required Libraries and Load Datasets
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid", font_scale=1.1)

# Load datasets
path = "/Users/vanessahuo/Documents/GitHub/maison-llf"

demographics = pd.read_csv(f"{path}/maison-llf-demographics.csv")
features = pd.read_csv(f"{path}/maison-llf-features.csv")

print("Demographics shape:", demographics.shape)
print("Features shape:", features.shape)

## Inspect DataFrame Structure and Data Types
Show the first rows, data types, and basic structure of both datasets.

In [ ]:
print("--- Demographics head ---")
print(demographics.head().to_string(index=False))
print("\n--- Demographics dtypes ---")
print(demographics.dtypes)
print("\n--- Features head ---")
print(features.head().to_string(index=False))
print("\n--- Features dtypes ---")
print(features.dtypes)

## Correlation Matrix and Heatmap
Compute correlations for numeric variables in the features dataset and display a heatmap.

In [ ]:
# Select numeric columns from features
numeric_features = features.select_dtypes(include=["number"])
print(f"Numeric feature columns: {numeric_features.shape[1]}")

# Compute correlation matrix
corr_matrix = numeric_features.corr()

# Plot heatmap for the numeric feature correlations
plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap="coolwarm", center=0, linewidths=0.5,
            cbar_kws={"shrink": 0.75}, fmt=".2f")
plt.title("Correlation Matrix Heatmap for Numeric Features")
plt.tight_layout()
plt.show()

## Missing Values and Percentage Missing
Calculate the count and percentage of missing values in each column for both datasets.

In [ ]:
for name, df in [("Demographics", demographics), ("Features", features)]:
    missing_count = df.isna().sum()
    missing_pct = df.isna().mean() * 100
    missing_summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_pct": missing_pct
    })
    missing_summary = missing_summary[missing_summary["missing_count"] > 0]
    print(f"--- {name} missing values ---")
    if missing_summary.empty:
        print("No missing values detected.")
    else:
        display(missing_summary.sort_values(by="missing_pct", ascending=False))
    print("\n")

## Histogram for Numeric Variable Distribution
Visualize the distribution of a representative numeric variable from the demographics dataset.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(demographics["age"], kde=True, bins=8, color="#2a9d8f")
plt.title("Age Distribution in Maison LLF Demographics")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Compare Categorical Group Means with a Boxplot
Use `sex` as the categorical grouping variable and compare the distribution of `age` across groups.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=demographics, x="sex", y="age", palette="Set2")
sns.swarmplot(data=demographics, x="sex", y="age", color="black", size=6, alpha=0.7)
plt.title("Age Distribution by Sex")
plt.xlabel("Sex")
plt.ylabel("Age")
plt.tight_layout()
plt.show()

print("Group means by sex:")
print(demographics.groupby("sex")["age"].mean().round(2))

## Patient-level SIS, OHS, and OKS Trend Plots
Plot patient-specific clinical assessment trajectories for SIS, OHS, and OKS items across assessment timestamps.

In [ ]:
# Convert clinical timestamp to datetime for plotting
features["clinical-timestamp"] = pd.to_datetime(features["clinical-timestamp"])

score_groups = {
    "SIS": [f"sis-{i:02d}" for i in range(1, 7)] + ["sis"],
    "OHS": [f"ohs-{i:02d}" for i in range(1, 13)] + ["ohs"],
    "OKS": [f"oks-{i:02d}" for i in range(1, 13)] + ["oks"]
}

for participant_id, participant_df in features.groupby("participant"):
    participant_df = participant_df.sort_values("clinical-timestamp")
    print(f"\nPatient {participant_id}: {participant_df['participant'].iloc[0]} rows = {participant_df.shape[0]}")

    for score_name, score_cols in score_groups.items():
        plt.figure(figsize=(10, 5))
        for col in score_cols:
            if col in participant_df.columns:
                plt.plot(
                    participant_df["clinical-timestamp"],
                    participant_df[col],
                    marker="o",
                    label=col,
                    linewidth=2,
                    alpha=0.85
                )

        plt.title(f"Patient {participant_id} {score_name} Trajectories")
        plt.xlabel("Clinical Timestamp")
        plt.ylabel("Score")
        plt.xticks(rotation=45)
        plt.legend(loc="best", fontsize="small", ncol=2)
        plt.tight_layout()
        plt.show()
